# 01 - Dataset preparation

**Required Kaggle settings**

| Setting | Value |
| --- | --- |
| Internet | **On** |
| Accelerator | **None** |

This is the only notebook permitted to touch the network. The GPU training notebooks run
with internet disabled, so everything that needs to be downloaded, verified, converted or
resized happens here exactly once.

**Output:** a versioned Kaggle dataset containing (a) the prepared frames/clips, (b) a
`manifest.json` recording every record and its split, and (c) a copy of this repository's
`src/` tree so the offline notebooks can import `wellbeing` without a clone.


In [ ]:
# Fail fast if the notebook is misconfigured, before any long download starts.
import socket, subprocess, sys
from pathlib import Path

def has_internet(host='raw.githubusercontent.com', port=443, timeout=5):
    try:
        socket.create_connection((host, port), timeout=timeout)
        return True
    except OSError:
        return False

assert has_internet(), 'Turn Internet ON in the notebook settings; this notebook cannot run offline.'
print('internet: ok')

try:
    import torch
    print('accelerator:', 'GPU (not needed here - free it for notebooks 02-04)' if torch.cuda.is_available() else 'CPU (correct)')
except ImportError:
    print('torch absent; not required for preparation')


In [ ]:
# Pull the repo so the notebook uses the committed, reviewed preparation code rather than
# a divergent copy pasted into a cell.
REPO_URL = 'https://github.com/PARTHG0106/EmotionSense-Extended.git'
REPO_DIR = Path('/kaggle/working/EmotionSense-Extended')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)

sys.path.insert(0, str(REPO_DIR / 'src'))
sys.path.insert(0, str(REPO_DIR / 'kaggle' / 'prep'))

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml'], check=True)

from prepare_datasets import (
    DatasetSpec, assign_splits, build_manifest, downscale_only, package, sha256_of, verify,
)
print('prep helpers loaded from', REPO_DIR / 'kaggle' / 'prep')


## Choose what to prepare

Specs live in `kaggle/prep/specs/`. Prepare one dataset per notebook run: mixing tasks in a
single Kaggle dataset makes the version history unreadable and forces the GPU notebooks to
mount data they do not use.


In [ ]:
SPEC_PATH = REPO_DIR / 'kaggle' / 'prep' / 'specs' / 'fall_lowres.yaml'
OUT_DIR = Path('/kaggle/working/fall_lowres')
RAW_DIR = Path('/kaggle/working/raw')
OUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

spec = DatasetSpec.load(SPEC_PATH)
print(f'{spec.name} v{spec.version}')
print(f'target short side: {spec.target_short_side}px, upscaling allowed: {spec.max_short_side_upscale}')
print(f'split by: {spec.split_by}, seed: {spec.seed}')
for source in spec.sources:
    print(f"  - {source['name']} [{source.get('license')}] {source.get('note', '')}")


## Download and verify

Most elderly-care and fall datasets require a signed research agreement and are **not**
directly downloadable. Register the credentialed ones as private Kaggle datasets once, then
list them in `MOUNTED_SOURCES`; only openly downloadable archives go in `DIRECT_SOURCES`.

Checksums are verified even for mounted data. A silently truncated archive trains and
validates cleanly on the truncated subset, which is the worst failure mode available.


In [ ]:
# name -> (url, sha256 or None)
DIRECT_SOURCES = {
    # 'le2i_fall.zip': ('https://example.org/le2i.zip', 'abc123...'),
}

# Datasets attached via 'Add Data' because they need a research agreement.
MOUNTED_SOURCES = {
    # 'toyota_smarthome': Path('/kaggle/input/toyota-smarthome-untrimmed'),
}

for filename, (url, expected) in DIRECT_SOURCES.items():
    target = RAW_DIR / filename
    if not target.exists():
        subprocess.run(['curl', '-L', '--fail', '-o', str(target), url], check=True)
    verify(target, expected)
    if expected is None:
        print(f'{filename} sha256={sha256_of(target)}  <- paste into the spec to pin it')

for name, mount in MOUNTED_SOURCES.items():
    assert mount.exists(), f'attach {name} via Add Data, or drop it from the spec'
    print(f'{name}: mounted at {mount}')

if not DIRECT_SOURCES and not MOUNTED_SOURCES:
    print('No sources configured. The cells below will run on a small synthetic set so you can')
    print('validate the whole path end to end before committing to multi-hour downloads.')


## Convert, filter and resize

Two filters here do most of the work for deployment realism:

1. **`min_bbox_pixels`** drops subjects too small for pose to be meaningful. Keeping them
   teaches the model to guess posture from a 12-pixel blob.
2. **`downscale_only`** never upscales. See the spec notes for why.


In [ ]:
import json, random

records = []

def add_record(clip_path, subject_id, label, width, height, fps, n_frames, source_name):
    """Append one prepared record. Keep provenance on every row."""
    if min(width, height) < spec.min_bbox_pixels:
        return
    out_w, out_h = downscale_only(width, height, spec.target_short_side, spec.max_short_side_upscale)
    records.append({
        'path': str(clip_path),
        'subject_id': str(subject_id),
        'label': label,
        'width': out_w,
        'height': out_h,
        'source_width': width,
        'source_height': height,
        'fps': fps,
        'n_frames': n_frames,
        'source': source_name,
    })

# --- Per-source extraction goes here. -----------------------------------------------
# Each dataset has its own annotation format (Le2i uses per-video text files, UP-Fall uses
# CSV, Toyota Smarthome uses JSON), so conversion cannot be generic. Convert each into
# add_record() calls and the rest of the pipeline is shared.

if not DIRECT_SOURCES and not MOUNTED_SOURCES:
    rng = random.Random(spec.seed)
    labels = ['fall', 'negative_sit', 'negative_lie', 'negative_bend', 'negative_walk']
    for subject in range(12):
        for index in range(20):
            # 1:5 positive ratio, deliberately imbalanced like a real home.
            label = 'fall' if index % 5 == 0 else rng.choice(labels[1:])
            add_record(
                clip_path=f'synthetic/s{subject:02d}_c{index:03d}.mp4',
                subject_id=f'subject_{subject:02d}',
                label=label,
                width=rng.choice([640, 720, 1280]),
                height=rng.choice([480, 576, 720]),
                fps=rng.choice([15, 25, 30]),
                n_frames=rng.randint(48, 200),
                source_name='synthetic',
            )

print(f'{len(records)} records, {len({r["subject_id"] for r in records})} subjects')


In [ ]:
# Subject-disjoint split assignment, then label remapping through the spec.
assignment = assign_splits([r['subject_id'] for r in records], spec.seed)
for record in records:
    record['split'] = assignment[record['subject_id']]
    if spec.label_map:
        record['label'] = spec.label_map.get(record['label'], record['label'])

manifest_path = build_manifest(spec, OUT_DIR, records)
manifest = json.loads(manifest_path.read_text())
for key in sorted(manifest['counts']):
    print(f"{key:<32} {manifest['counts'][key]}")


## Sanity checks that must pass before uploading

These are assertions rather than printouts on purpose. A dataset with a leaked subject or a
class missing from validation produces results that look fine and mean nothing.


In [ ]:
from collections import Counter

by_split = {}
for record in records:
    by_split.setdefault(record['split'], set()).add(record['subject_id'])

# 1. No subject may appear in two splits.
for a in by_split:
    for b in by_split:
        if a < b:
            overlap = by_split[a] & by_split[b]
            assert not overlap, f'subject leak between {a} and {b}: {sorted(overlap)[:5]}'
print('subject disjointness: ok', {k: len(v) for k, v in by_split.items()})

# 2. Every label must be represented in train and val, or the metric is undefined for it.
labels_seen = {split: Counter(r['label'] for r in records if r['split'] == split) for split in by_split}
all_labels = set(r['label'] for r in records)
for split in ('train', 'val'):
    missing = all_labels - set(labels_seen.get(split, {}))
    assert not missing, f'{split} is missing labels {missing}; re-split or gather more subjects'
print('label coverage: ok')

# 3. Nothing was upscaled.
upscaled = [r for r in records if min(r['width'], r['height']) > min(r['source_width'], r['source_height'])]
assert not upscaled, f'{len(upscaled)} records were upscaled; check downscale_only usage'
print('no upscaling: ok')

# 4. Report the true class ratio so nobody mistakes a balanced test set for reality.
positives = sum(1 for r in records if r['label'] == 'fall')
print(f'positive ratio: {positives}/{len(records)} = {positives / max(1, len(records)):.3f}')


In [ ]:
# Bundle the repo source into the dataset. This is how notebooks 02-04 import `wellbeing`
# with no network: Kaggle datasets are the only way code reaches an offline GPU session.
import shutil

code_dir = OUT_DIR / 'code'
if code_dir.exists():
    shutil.rmtree(code_dir)
code_dir.mkdir(parents=True)
shutil.copytree(REPO_DIR / 'src', code_dir / 'src')
shutil.copytree(REPO_DIR / 'kaggle' / 'train', code_dir / 'train')
shutil.copytree(REPO_DIR / 'configs', code_dir / 'configs')

commit = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True
).stdout.strip()
(code_dir / 'COMMIT').write_text(commit + chr(10))
print('bundled repo at commit', commit)


In [ ]:
# Package and upload. Requires kaggle.json in Kaggle Secrets or ~/.kaggle/.
# Use `kaggle datasets version` for subsequent revisions so training runs can pin a version.
SLUG = f"{spec.name.replace('_', '-')}-v{spec.version}"
archive = package(OUT_DIR, SLUG)
print('archive:', archive)
print()
print('First upload:')
print(f'  kaggle datasets create -p {OUT_DIR} --dir-mode zip')
print('Later revisions:')
print(f'  kaggle datasets version -p {OUT_DIR} -m "{spec.name} v{spec.version}" --dir-mode zip')


## Handoff checklist

Before opening notebook 02, confirm:

- [ ] All four sanity assertions above passed.
- [ ] `manifest.json` is present in the uploaded dataset and its counts look right.
- [ ] Checksums for every source are pinned in the spec, and the spec change is committed.
- [ ] `code/COMMIT` matches the repo commit you intend to train from.
- [ ] Any pretrained backbone weights are uploaded as their **own** Kaggle dataset; the GPU
      notebook cannot download them.
